In [ ]:
import pandas as pd
import glob
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV


In [2]:
# Definir ruta donde estan los archivos
path = r'../data/raw/'

In [3]:
# Listar archivos Parquet que comienzan con 'data'
all_files = glob.glob(os.path.join(path, 'data*.parquet'))

In [4]:
# Leer cada archivo Parquet y almacenarlo en una lista de DataFrames
df_list = []
for filename in all_files:
    df_part = pd.read_parquet(filename, engine='pyarrow')
    df_list.append(df_part)

In [5]:
# Concatenar todos los DataFrames en un único DataFrame
df = pd.concat(df_list, axis=0, ignore_index=True)

In [6]:
# Convertimos Date a datetime (necesario para split temporal)
df['Date'] = pd.to_datetime(df['Date'])

/var/folders/rf/90l5_42s7vn54w43kvkc8g9c0000gn/T/ipykernel_5440/1063807796.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['Date'] = pd.to_datetime(df['Date'])


1.1 Feature 1 — Rendimiento 5 días

In [ ]:
df['Price_Change_5d']

0         0.025553
1         0.009218
2        -0.026367
3        -0.067658
4        -0.030910
            ...   
620090    0.002410
620091    0.001953
620092    0.028600
620093    0.015769
620094    0.005231
Name: Price_Change_5d, Length: 620095, dtype: float64

1.2 Feature 2 — Precio vs media 20 días

In [ ]:
df['price_vs_sma20'] = (df['Close'] / df['SMA_20']) - 1

1.3 Feature 3 — Fuerza del movimiento (RSI categórico)

In [9]:
def rsi_strength(rsi):
    if rsi < 40:
        return 0   # Débil
    elif rsi <= 60:
        return 1   # Neutral
    else:
        return 2   # Fuerte

df['rsi_strength'] = df['RSI'].apply(rsi_strength)


1.4 Feature 4 — Volumen relativo

In [ ]:
df['Volume_Ratio']

0         0.941486
1         1.264012
2         0.523449
3         1.460278
4         1.027248
            ...   
620090    0.865129
620091    0.961570
620092    1.397328
620093    1.497178
620094    2.580893
Name: Volume_Ratio, Length: 620095, dtype: float64

1.5 Feature 5 — Volatilidad categórica

In [11]:
v1 = df['Volatility'].quantile(0.33)
v2 = df['Volatility'].quantile(0.66)

def vol_level(v):
    if v < v1:
        return 0   # Baja
    elif v < v2:
        return 1   # Media
    else:
        return 2   # Alta

df['vol_level'] = df['Volatility'].apply(vol_level)


1.6 Feature 6 — Riesgo de mercado

In [ ]:
# Por ahora simulamos riesgo medio (1)
df['market_risk'] = 1

2.1 Target 

In [13]:
# target = 1 si Future_Return_10d >= 2%
# target = 0 si Future_Return_10d <= 0%
df['target'] = np.nan

df.loc[df['Future_Return_10d'] >= 0.02, 'target'] = 1
df.loc[df['Future_Return_10d'] <= 0.0, 'target'] = 0

df = df.dropna(subset=['target'])

2.2 Matriz final

In [14]:
features_simple = ['Price_Change_5d',
                   'price_vs_sma20',
                   'rsi_strength',
                   'Volume_Ratio',
                   'vol_level',
                   'market_risk']

X = df[features_simple]
y = df['target']


In [15]:
train_idx = []
test_idx = []

for tkr, g in df.groupby('Ticker'):
    g = g.sort_values('Date')
    split = int(len(g) * 0.8)
    train_idx.extend(g.index[:split])
    test_idx.extend(g.index[split:])


In [16]:
X_train = X.loc[train_idx]
y_train = y.loc[train_idx]

X_test = X.loc[test_idx]
y_test = y.loc[test_idx]


In [ ]:
# Instanciar el modelo con valores por defecto
# Solo mantenemos random_state para que los resultados sean repetibles
# y class_weight='balanced' porque los datos de bolsa suelen estar desbalanceados.
simple_rf = RandomForestClassifier(random_state=18, 
                                   class_weight='balanced')

# Entrenar directamente con los datos originales
simple_rf.fit(X_train, y_train)

# Evaluar
y_pred = simple_rf.predict(X_test)
y_probs = simple_rf.predict_proba(X_test)[:, 1]

# RESULTADOS MODELO BASE (SIN OPTIMIZAR)
f'ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}'


--- RESULTADOS MODELO BASE (SIN OPTIMIZAR) ---
ROC-AUC Score: 0.5100
              precision    recall  f1-score   support

         0.0       0.54      0.62      0.58     56400
         1.0       0.47      0.39      0.42     48138

    accuracy                           0.51    104538
   macro avg       0.51      0.51      0.50    104538
weighted avg       0.51      0.51      0.51    104538



In [ ]:
classification_report(y_test, y_pred)

## Hiperparametrización

In [ ]:
# Optimización con RandomizedSearchCV
# Usamos directamente X_train y y_train (sin escalas)
param_dist = {'n_estimators': [100, 200, 300, 500],
              'max_depth': [None, 10, 20, 30],
              'min_samples_leaf': [50, 100, 200],
              'max_features': ['sqrt', 'log2'],
              'bootstrap': [True]}

random_search = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=18, class_weight='balanced'),
                                   param_distributions=param_dist,
                                   n_iter=20,
                                   cv=5, 
                                   scoring='roc_auc',
                                   n_jobs=-1,
                                   verbose=1,
                                   random_state=18)

# Entrenamos con los datos originales
random_search.fit(X_train, y_train)

# Evaluación del Modelo 
best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_probs = best_rf.predict_proba(X_test)[:, 1]

f'Mejor configuración: {random_search.best_params_}'


Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:
f'ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}'

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
'''PASO 3: Guardar
if not os.path.exists('models'):
    os.makedirs('models')

joblib.dump(best_rf, '../models/best_rf_no_scaling.pkl')'''

In [ ]:
coef = pd.Series(best_rf.coef_[0],
                 index=features_simple).sort_values()

coef


Price_Change_5d   -0.024934
rsi_strength      -0.010950
market_risk       -0.001213
price_vs_sma20     0.007620
vol_level          0.014888
Volume_Ratio       0.027061
dtype: float64